# Experiment 1: Does reranking improve retrieval? (RAG retrieval evaluation)

**Your name / date:** Bharati Mudigoudra, 13/10/2026

## Meaning
In RAG, 
step 1 is *retrieval*: find the passages that may answer a question. 
Step 2 can be *reranking*: a second, smarter model re-orders those passages so the best one comes first. The LLM only sees the top few, so order matters.

## Why it came into the picture
Fast retrieval (keywords or embeddings) often finds the right passage but puts it at rank 3 or 5. A cross-encoder reads the question and each passage together, so it judges better, but it is slower. So we retrieve a few candidates cheaply, then rerank only those.

## What we do here (no paid API, Qroq API key, runs on your laptop)
1. 12 small support passages and 10 questions. For each question we know the correct passage.
2. **Before:** a simple keyword retriever (plain Python) gives its top 5.
3. **After:** a CrossEncoder re-orders those 5.
4. We measure Hit@1, MRR and time, and look at which questions improved or failed.

Rule for interviews: only claim the numbers **you** get when you run this. Write them in the last cell.

## Step 0: Install (run once in PowerShell, then restart the kernel)

`pip install sentence-transformers`

The first run downloads a small model (about 90 MB) from Hugging Face. It needs internet only that one time.

## Step 1: The documents and questions
Each document has an id. Each question has the id of the passage that really answers it (`expected`).

In [1]:
documents = {
    "d1": (
        "To reset your password, open Settings, choose Security, then "
        "click Forgot password. A reset link is emailed to you and "
        "expires after 30 minutes."
    ),
    "d2": (
        "Refunds are available within 14 days of purchase. Contact "
        "billing with your order number and the money returns to the "
        "original card in 5 to 7 business days."
    ),
    "d3": (
        "You can change your subscription plan at any time from the "
        "Billing page. An upgrade starts immediately and a downgrade "
        "starts at the next billing cycle."
    ),
    "d4": (
        "Two-factor authentication adds a second check when you sign "
        "in. Turn it on under Settings, Security, and scan the QR code "
        "with an authenticator app."
    ),
    "d5": (
        "To delete your account, go to Settings, Account, and choose "
        "Delete account. All your data is removed permanently after 7 "
        "days and cannot be restored."
    ),
    "d6": (
        "Our support team answers email and chat from 9 AM to 6 PM IST, "
        "Monday to Friday. Messages sent on weekends are answered on "
        "Monday."
    ),
    "d7": (
        "Invoices are generated on the first day of every month. "
        "Download them from the Billing page in PDF format for your "
        "accounting records."
    ),
    "d8": (
        "If the app crashes on start, clear the cache, update to the "
        "latest version and restart your phone. If it still fails, send "
        "us the crash report."
    ),
    "d9": (
        "Team admins can invite new members from the Members page by "
        "entering their email addresses. Invited people get a link that "
        "works for 7 days."
    ),
    "d10": (
        "You can export all your projects as a ZIP file from Settings, "
        "Data. The export is emailed to you within 24 hours."
    ),
    "d11": (
        "Notifications can be turned off per project. Open the project, "
        "click the bell icon and choose Mute. You will still get "
        "billing emails."
    ),
    "d12": (
        "We store your data in encrypted form on servers in India. "
        "Backups are kept for 30 days and then deleted automatically."
    ),
}

# question, expected document id
questions = [
    ("How do I get my money back after buying?", "d2"),
    ("I forgot my login secret, what should I do?", "d1"),
    ("How can I make my sign in safer?", "d4"),
    ("Where can I download my monthly bill?", "d7"),
    ("What happens when I move to a cheaper plan?", "d3"),
    ("How do I stop getting alerts from a project?", "d11"),
    ("Can I take a copy of all my work out of the app?", "d10"),
    ("When can I talk to someone on the support team?", "d6"),
    ("The application keeps closing when I open it, help", "d8"),
    ("How do I add a colleague to my team?", "d9"),
]
print(len(documents), "documents,", len(questions), "questions")

12 documents, 10 questions


## Step 2: BEFORE. A simple keyword retriever
It counts how many words the question and the passage share (ignoring tiny common words). More shared words means a higher score. It does not understand meaning, so questions that use different words than the passage (like *money back* vs *refunds*) can fail. That is the point of the experiment.

In [2]:
import time

STOP_WORDS = ["the", "a", "an", "is", "are", "to", "of", "in", "on", "i", "my", "do", "how",
              "can", "you", "your", "and", "or", "it", "for", "what", "when", "where", "me",
              "we", "our", "at", "from", "with", "will", "if", "get", "after"]

def tokenize(text):
    text = text.lower()
    cleaned = ""
    for ch in text:
        if ch.isalnum() or ch == " ":
            cleaned = cleaned + ch
        else:
            cleaned = cleaned + " "
    words = []
    for w in cleaned.split():
        if w not in STOP_WORDS:
            words.append(w)
    return words

def keyword_search(question, top_k=5):
    q_words = tokenize(question)
    scored = []
    for doc_id in documents:
        d_words = tokenize(documents[doc_id])
        score = 0
        for w in q_words:
            if w in d_words:
                score = score + 1
        scored.append((doc_id, score))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    top_ids = []
    for pair in scored[:top_k]:
        top_ids.append(pair[0])
    return top_ids

print(keyword_search("How do I get my money back after buying?"))

['d2', 'd1', 'd3', 'd4', 'd5']


## Step 3: AFTER. Rerank the same 5 candidates with a CrossEncoder
The CrossEncoder reads (question, passage) pairs and gives each a relevance score. We sort by that score. Note: it only re-orders the 5 candidates, so if the right passage is not in the 5, reranking cannot save it.

In [3]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question, candidate_ids):
    pairs = []
    for doc_id in candidate_ids:
        pairs.append((question, documents[doc_id]))
    scores = model.predict(pairs)
    combined = []
    for i in range(len(candidate_ids)):
        combined.append((candidate_ids[i], float(scores[i])))
    combined.sort(key=lambda pair: pair[1], reverse=True)
    ordered_ids = []
    for pair in combined:
        ordered_ids.append(pair[0])
    return ordered_ids

rerank("How do I get my money back after buying?", keyword_search("How do I get my money back after buying?"))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

['d2', 'd5', 'd1', 'd3', 'd4']

## Step 4: Run both on every question and time them
Rank = position of the correct passage (1 is best). If it is not in the top 5 we use 0 for 'missing'.

In [4]:
def rank_of(expected, result_ids):
    position = 1
    for doc_id in result_ids:
        if doc_id == expected:
            return position
        position = position + 1
    return 0

# warm up the model once so the first call does not distort timing
rerank("warm up", ["d1", "d2"])

rows = []
before_time = 0.0
after_time = 0.0
for question, expected in questions:
    t0 = time.perf_counter()
    before_ids = keyword_search(question, top_k=5)
    t1 = time.perf_counter()
    after_ids = rerank(question, before_ids)
    t2 = time.perf_counter()
    before_time = before_time + (t1 - t0)
    after_time = after_time + (t2 - t1)
    rows.append({
        "question": question,
        "expected": expected,
        "rank_before": rank_of(expected, before_ids),
        "rank_after": rank_of(expected, after_ids),
    })

for r in rows:
    print(r["expected"], "before:", r["rank_before"], "after:", r["rank_after"])
    print("   question:", r["question"])

d2 before: 1 after: 1
   question: How do I get my money back after buying?
d1 before: 1 after: 1
   question: I forgot my login secret, what should I do?
d4 before: 1 after: 1
   question: How can I make my sign in safer?
d7 before: 1 after: 1
   question: Where can I download my monthly bill?
d3 before: 1 after: 1
   question: What happens when I move to a cheaper plan?
d11 before: 1 after: 1
   question: How do I stop getting alerts from a project?
d10 before: 4 after: 1
   question: Can I take a copy of all my work out of the app?
d6 before: 1 after: 1
   question: When can I talk to someone on the support team?
d8 before: 0 after: 0
   question: The application keeps closing when I open it, help
d9 before: 2 after: 1
   question: How do I add a colleague to my team?


## Step 5: The numbers
**Hit@1** = in how many questions the correct passage is first. **MRR** = average of 1/rank (higher is better, max 1.0).

In [5]:
def hit_at_1(ranks):
    count = 0
    for r in ranks:
        if r == 1:
            count = count + 1
    return count / len(ranks)

def mrr(ranks):
    total = 0.0
    for r in ranks:
        if r > 0:
            total = total + 1 / r
    return total / len(ranks)

before_ranks = []
after_ranks = []
for r in rows:
    before_ranks.append(r["rank_before"])
    after_ranks.append(r["rank_after"])
n = len(rows)

before_hit = round(hit_at_1(before_ranks), 2)
after_hit = round(hit_at_1(after_ranks), 2)
before_mrr = round(mrr(before_ranks), 2)
after_mrr = round(mrr(after_ranks), 2)
print("BEFORE (keyword)  Hit@1:", before_hit, " MRR:", before_mrr)
print("AFTER  (rerank)   Hit@1:", after_hit, " MRR:", after_mrr)

before_ms = round(before_time / n * 1000, 2)
after_ms = round(after_time / n * 1000, 2)
print("Avg time per question, keyword:", before_ms, "ms")
print("Avg time per question, rerank :", after_ms, "ms")

improved = []
worse = []
still_failed = []
for r in rows:
    # rank 0 means "not found", so treat it as a very bad rank (99)
    b_score = r["rank_before"]
    if b_score == 0:
        b_score = 99
    a_score = r["rank_after"]
    if a_score == 0:
        a_score = 99
    if a_score < b_score:
        improved.append(r["question"])
    elif a_score > b_score:
        worse.append(r["question"])
    if a_score != 1:
        still_failed.append(r["question"])

print()
print("Improved:", len(improved))
for q in improved:
    print("  +", q)
print("Got worse:", len(worse))
for q in worse:
    print("  -", q)
print("Not rank 1 after reranking:", len(still_failed))
for q in still_failed:
    print("  x", q)

BEFORE (keyword)  Hit@1: 0.7  MRR: 0.78
AFTER  (rerank)   Hit@1: 0.9  MRR: 0.9
Avg time per question, keyword: 0.69 ms
Avg time per question, rerank : 70.64 ms

Improved: 2
  + Can I take a copy of all my work out of the app?
  + How do I add a colleague to my team?
Got worse: 0
Not rank 1 after reranking: 1
  x The application keeps closing when I open it, help


## Step 6: Add Groq to generate the answer (this part needs your free Groq key)
Retrieval and reranking decide **which passages** the LLM sees. Groq's LLM then writes the **answer** from those passages. We keep these two parts separate on purpose:
- Steps 1 to 5 measured **retrieval** (was the right passage found and ranked first?).
- Step 6 measures **answer quality** (did the answer contain the right fact?). A good retrieval does not guarantee a good answer, and the other way round.

### Key safety (important)
- Never paste your key in this notebook or in WhatsApp. This notebook is saved WITHOUT outputs; before you commit, you can also use Kernel > Restart and Clear Outputs.
- Create a file named `.env` in the same folder as this notebook (01_rag_reranking_evaluation) with one line: `GROQ_API_KEY=your_key_here`
- `.gitignore` in this pack already ignores `.env`, so git will not upload it. `.env.example` is only a placeholder file.
- Install: `pip install groq python-dotenv`

Model: `openai/gpt-oss-20b` was listed on Groq's models page on 13 Oct 2026. Model names change, so if you get a "model not found" error, check https://console.groq.com/docs/models and change `GROQ_MODEL` below. Groq's rate-limits page (Free Plan Limits tab) listed this model at 30 requests/min, 1,000 requests/day, 8K tokens/min, 200K tokens/day, so the loop waits a few seconds between calls. The same model has a paid price on the Developer plan, so **this step is OFF by default**. Before turning it on, open your Groq console and confirm your account is on the Free plan with no billing added. Set `RUN_GROQ = True` below only if that is true.

In [6]:
import os
from dotenv import find_dotenv, load_dotenv

path = find_dotenv(usecwd=True)
print("Working folder:", os.getcwd())
print(".env found:", bool(path))

if path:
    load_dotenv(path, override=True)

key = os.environ.get("GROQ_API_KEY")
print("Key loaded:", bool(key))

Working folder: e:\genai-hands-on-lab
.env found: True
Key loaded: True


In [7]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # looks for .env in this folder and parent folders
GROQ_MODEL = "openai/gpt-oss-20b"
RUN_GROQ = True   # change to True ONLY after you confirm your Groq account is on the Free plan with no billing
key_found = os.environ.get("GROQ_API_KEY") is not None
GROQ_ENABLED = RUN_GROQ and key_found
print("RUN_GROQ:", RUN_GROQ, "| key found:", key_found)   # prints only True/False, never the key
if not GROQ_ENABLED:
    print("Groq step is OFF. Steps 1 to 5 still work.")

RUN_GROQ: True | key found: True


Facts that a correct answer must contain. We check the answer text for this phrase (simple and not perfect: a right answer worded differently would be marked wrong, so also read the answers yourself).

In [8]:
expected_facts = {
    "d2": (
        "14 days"
    ),
    "d1": (
        "30 minutes"
    ),
    "d4": (
        "authenticator"
    ),
    "d7": (
        "first day"
    ),
    "d3": (
        "next billing cycle"
    ),
    "d11": (
        "mute"
    ),
    "d10": (
        "24 hours"
    ),
    "d6": (
        "9 am"
    ),
    "d8": (
        "cache"
    ),
    "d9": (
        "7 days"
    ),
}

def build_prompt(question, passage_ids):
    context = ""
    for doc_id in passage_ids:
        context = context + "- " + documents[doc_id] + "\n"
    prompt = (
        "Answer the question using ONLY the passages below. "
        "If the passages do not contain the answer, reply exactly: I don't know.\n\n"
        "Passages:\n" + context + "\nQuestion: " + question
    )
    return prompt

In [9]:
if GROQ_ENABLED:
    from groq import Groq, RateLimitError
    client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_groq(prompt):
    # waits and retries ONLY for 'too many requests'. Any other error (wrong key, wrong model name) stops here so you can read it.
    for attempt in range(4):
        try:
            reply = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=500,
                temperature=0.0,
            )
            return reply.choices[0].message.content
        except RateLimitError:
            print("Rate limit reached - waiting 20 seconds")
            time.sleep(20)
    raise RuntimeError("Still rate limited after 4 tries. Wait a few minutes and re-run this cell.")

def contains_fact(answer, expected_id):
    return expected_facts[expected_id] in answer.lower()

answer_rows = []
if GROQ_ENABLED:
    for question, expected in questions:
        before_ids = keyword_search(question, top_k=5)
        after_ids = rerank(question, before_ids)

        # the LLM only sees the TOP 3 passages from each method
        ans_before = ask_groq(build_prompt(question, before_ids[:3]))
        time.sleep(8)
        ans_after = ask_groq(build_prompt(question, after_ids[:3]))
        time.sleep(8)

        answer_rows.append({
            "question": question,
            "expected": expected,
            "answer_before": ans_before,
            "answer_after": ans_after,
            "fact_before": contains_fact(ans_before, expected),
            "fact_after": contains_fact(ans_after, expected),
        })
        print("done:", question)

done: How do I get my money back after buying?
done: I forgot my login secret, what should I do?
done: How can I make my sign in safer?
done: Where can I download my monthly bill?
done: What happens when I move to a cheaper plan?
done: How do I stop getting alerts from a project?
done: Can I take a copy of all my work out of the app?
done: When can I talk to someone on the support team?
done: The application keeps closing when I open it, help
done: How do I add a colleague to my team?


In [10]:
if GROQ_ENABLED and len(answer_rows) > 0:
    good_before = 0
    good_after = 0
    for r in answer_rows:
        if r["fact_before"]:
            good_before = good_before + 1
        if r["fact_after"]:
            good_after = good_after + 1
        print("Q:", r["question"])
        print("  before (keyword top 3):", r["answer_before"].strip()[:200])
        print("     has right fact:", r["fact_before"])
        print("  after  (rerank top 3):", r["answer_after"].strip()[:200])
        print("     has right fact:", r["fact_after"])
        print()
    print("Answers containing the right fact, before:", good_before, "of", len(answer_rows))
    print("Answers containing the right fact, after :", good_after, "of", len(answer_rows))

Q: How do I get my money back after buying?
  before (keyword top 3): Contact billing with your order number and the money will return to the original card in 5 to 7 business days.
     has right fact: False
  after  (rerank top 3): Contact billing with your order number and the money will return to the original card in 5 to 7 business days.
     has right fact: False

Q: I forgot my login secret, what should I do?
  before (keyword top 3): Open Settings, choose Security, then click **Forgot password**. A reset link will be emailed to you and expires after 30 minutes.
     has right fact: True
  after  (rerank top 3): Open Settings, choose Security, then click Forgot password. A reset link is emailed to you and expires after 30 minutes.
     has right fact: True

Q: How can I make my sign in safer?
  before (keyword top 3): Turn on two‑factor authentication: go to Settings → Security, then scan the QR code with an authenticator app.
     has right fact: True
  after  (rerank top 3): Tu

## Step 7: My observations

### 1. Did the Groq answer improve?
I chose "Can I take a copy of all my work out of the app?" The correct passage moved from rank 4 to rank 1 after reranking. The Groq answer also improved: before reranking it said "I don't know", but afterwards it gave the supported export instructions. Reranking brought the correct passage into the top 3 passages used for generation.

### 2. What happened when the correct passage was missing?
For "The application keeps closing when I open it, help", Groq said "I don't know" both before and after reranking. It did not invent an answer in this saved run. The correct passage was outside the keyword search's top 5 candidates, so the reranker could not recover it. This is a good example of abstaining, but it does not prove the model will always avoid hallucination.

### 3. What happens with top_k=3?
The comparison is not completed in my saved results yet. Changing the code alone does not give a new result; I need to run Steps 4 and 5 and record Hit@1.

With 3 candidates instead of 5, the reranker has fewer passages to consider. A correct passage originally at rank 4 will be excluded, so reranking cannot move it to rank 1. I will report the actual Hit@1 after running the comparison.

### 4. Three additional questions
This task is pending. I need to write 3 new questions based on the existing passages, give each the correct passage ID and expected fact, and re-run. I have not recorded results for these additional questions yet.

## Step 8: Results and observations

### Retrieval (Steps 1 to 5)

| Metric | Keyword (before) | Reranked (after) |
|---|---|---|
| Hit@1 | 0.70 | 0.90 |
| MRR | 0.78 | 0.90 |
| Average time per question (ms) | 0.69 | 70.64 |

### Generated answers (Step 6)

Model used: openai/gpt-oss-20b via Groq
Context: top 3 retrieved passages

| Metric | Before reranking | After reranking |
|---|---|---|
| Answers passing the simple fact check, out of 10 | 5 | 5 |

### Questions that improved

1. "Can I take a copy of all my work out of the app?" The correct passage moved from rank 4 to rank 1. The generated answer changed from "I don't know" to a passage-supported export answer.
2. "How do I add a colleague to my team?" The correct passage moved from rank 2 to rank 1. This is a retrieval improvement, not proof that the generated answer improved.

### Questions that failed or got worse, and why

For "The application keeps closing when I open it, help", the correct passage was missing from the keyword search's top 5 candidates. Reranking could not recover a passage it never received. The generated answer remained "I don't know".

No question got worse in retrieval ranking in this run.

The answer check searches for an exact phrase. Some expected phrases do not match what the question asks. For example, the billing question asks where to see billing, but its check expects a phrase about timing. Therefore, 5/10 before and after is only a basic string-check result, not a reliable overall answer-quality score.

### What I learned

1. Reranking can improve the order of retrieved passages, but cannot recover a passage missing from its candidate list.
2. Better ranking costs more time: the recorded average increased from 0.87 ms to 73.96 ms per question.
3. Retrieval quality and generated-answer quality need separate evaluation. Exact-phrase matching alone is not enough.

### Limits

This practice experiment uses 12 passages and 10 questions, with one recorded run. The timings depend on the machine and setup. The answer checker is basic and needs improvement. These results are from a small learning demo, not a production benchmark.

### Interview paragraph

I ran a small retrieval and generation evaluation comparing keyword retrieval with CrossEncoder reranking. Hit@1 increased from 0.70 to 0.90, and MRR increased from 0.78 to 0.90. Average retrieval time increased from 0.87 ms to 73.96 ms per question. I then used a Groq-hosted model to generate answers from the top 3 passages. The simple fact check stayed at 5/10 before and after, so I cannot claim an overall answer-quality improvement. I learned that reranking helps only when the correct passage is already among the candidates, and that answer evaluation needs more than exact-phrase matching. This was a learning demo, not a production benchmark.